In [ ]:
from torchvision import models
print(models.ResNet18_Weights.DEFAULT.transforms())


In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]     
imagenet_std  = [0.229, 0.224, 0.225]
img_size = 64 

In [ ]:

pre_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.Resize(img_size),                     
    transforms.Grayscale(num_output_channels=3),     
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
pre_eval_transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

pre_train_full = datasets.FashionMNIST(root='./data', train=True,  download=True, transform=pre_train_transform)
pre_eval_full  = datasets.FashionMNIST(root='./data', train=True,  download=True, transform=pre_eval_transform)
pre_test_data  = datasets.FashionMNIST(root='./data', train=False, download=True, transform=pre_eval_transform)

pre_train_set      = Subset(pre_train_full, train_indices)          # same 48k images, augmented
pre_train_eval_set = Subset(pre_eval_full,  train_indices[:10000])  
pre_val_set        = Subset(pre_eval_full,  val_indices)            # same 12k validation images, clean

print(pre_train_set[0][0].shape)   


In [ ]:
class PretrainedResNet(nn.Module):

    def __init__(self, n_fc=2, hidden_units=256):
    
        super(PretrainedResNet, self).__init__()
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
         
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])   # output: (N, 512, 1, 1)
        for param in self.backbone.parameters():
            param.requires_grad = False

        layers = [] 
        in_dim = 512
        for _ in range(n_fc - 1):
            layers += [nn.Linear(in_dim, hidden_units), nn.ReLU()]
            in_dim = hidden_units
        layers.append(nn.Linear(in_dim, 10))          # 输出层：输出 10 个类别的分数（logits）
        self.head = nn.Sequential(*layers)

    def forward(self, x):
        # x: (N, 3, 64, 64)
        with torch.no_grad():                         
            x = self.backbone(x)                      # Shape: (N, 512, 1, 1)
        x = x.view(x.size(0), -1)                     # Shape: (N, 512)
        return self.head(x)                           # Shape: (N, 10)

for n_fc in [1, 2, 3]:
    m = PretrainedResNet(n_fc)
    n_train = sum(p.numel() for p in m.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in m.parameters())
    print(f'{n_fc} FC layer(s): trainable {n_train:,} / total {n_total:,}')



In [ ]:
# The backbone never changes -> val / test / train-eval features are computed ONCE
feature_net = PretrainedResNet().to(device)
feature_net.eval()

def extract_features(dataset):
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    feats, labs = [], []
    with torch.no_grad():
        for images, labels in loader:
            f = feature_net.backbone(images.to(device))      # (N, 512, 1, 1)
            feats.append(f.view(f.size(0), -1).cpu())        # (N, 512)
            labs.append(labels)
    return torch.cat(feats), torch.cat(labs)

start = time.time()
val_feats        = extract_features(pre_val_set)
test_feats       = extract_features(pre_test_data)
train_eval_feats = extract_features(pre_train_eval_set)
extract_time = time.time() - start
print(f'feature extraction (one-time): {extract_time:.1f} s')

def head_accuracy(head, feats):
    x, y = feats
    head.eval()
    with torch.no_grad():
        _, predicted = torch.max(head(x.to(device)), dim=1)
    return (predicted.cpu() == y).float().mean().item()

In [ ]:
criterion = nn.CrossEntropyLoss()

def train_pretrained(n_fc, lr=0.05, num_epochs=10, batch_size=64, hidden_units=256):
    torch.manual_seed(2026)                               # same head init, batch order, augmentations
    model = PretrainedResNet(n_fc, hidden_units).to(device)
    optimizer = optim.SGD(model.head.parameters(), lr=lr, momentum=0)   
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    loader = DataLoader(pre_train_set, batch_size=batch_size, shuffle=True)

    history = {'train': [], 'val': [], 'test': [], 'epoch time': []}
    for epoch in range(num_epochs):
        # ===== Training phase (timed) =====
        start = time.time()
        model.train()
        model.backbone.eval()          # keep BatchNorm running stats frozen too
        for images, labels in loader:
            # 1. to device
            images = images.to(device)
            labels = labels.to(device)
            # 2. zero gradients
            optimizer.zero_grad()
            # 3. forward
            outputs = model(images)
            # 4. loss
            loss = criterion(outputs, labels)
            # 5. backward (gradients only reach the FC layers)
            loss.backward()
            # 6. update the FC layers
            optimizer.step()
        history['epoch time'].append(time.time() - start)

        # ===== Evaluation (head only, on cached features) =====
        history['train'].append(head_accuracy(model.head, train_eval_feats))
        history['val'].append(head_accuracy(model.head, val_feats))
        history['test'].append(head_accuracy(model.head, test_feats))
        print(f"  Epoch [{epoch+1:2d}/{num_epochs}] - Train Acc: {history['train'][-1]:.4f}, "
              f"Val Acc: {history['val'][-1]:.4f}, Test Acc: {history['test'][-1]:.4f}, "
              f"Time: {history['epoch time'][-1]:.1f} s")
    return model, history, n_trainable

In [ ]:
fc_rows = []
fc_history = {}
for n_fc in [1, 2, 3]:
    print(f'\n=== {n_fc} FC layer(s) ===')
    model_pre, history, n_trainable = train_pretrained(n_fc)
    fc_history[n_fc] = history
    fc_rows.append({'FC layers':        n_fc,
                    'trainable params': n_trainable,
                    'train acc (clean)': history['train'][-1],
                    'val acc':          history['val'][-1],
                    'best val acc':     max(history['val']),
                    'test acc':         history['test'][-1],
                    'time / epoch (s)': np.mean(history['epoch time']),
                    'total time (s)':   np.sum(history['epoch time'])})

fc_df = pd.DataFrame(fc_rows)
fc_df.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs = range(1, 11)
for ax, n_fc in zip(axes, fc_history):
    ax.plot(epochs, fc_history[n_fc]['train'], marker='o', label='train (clean)')
    ax.plot(epochs, fc_history[n_fc]['val'],   marker='o', label='validation')
    ax.plot(epochs, fc_history[n_fc]['test'],  marker='o', label='test')
    ax.set_title(f'ResNet-18 (frozen) + {n_fc} FC layer(s)')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
best_n_fc = int(fc_df.loc[fc_df['best val acc'].idxmax(), 'FC layers'])   # chosen by VALIDATION
best_row = fc_df[fc_df['FC layers'] == best_n_fc].iloc[0]

# MLP (Q5) and CNN (Q7) numbers from the cells above, all trained with the Q5 augmentation
comparison_q8 = pd.DataFrame([
    {'model': 'MLP 2x256 + L2 (Q5)',  'trainable params': 269322,
     'test acc': 0.8245, 'total train time (s)': 38.4},
    {'model': 'CNN 32/64, 3x3 (Q7)',  'trainable params': 824458,
     'test acc': 0.8966, 'total train time (s)': 57.2},
    {'model': f'ResNet-18 frozen + {best_n_fc} FC (Q8)', 'trainable params': best_row['trainable params'],
     'test acc': best_row['test acc'], 'total train time (s)': best_row['total time (s)']},
])
comparison_q8

In [ ]:
# Q9 step 1: CIFAR-10 -- natural RGB photos, 32x32, 10 classes (50k train / 10k test)
# Same preprocessing + Q5 augmentation as Q8. only difference: already RGB 
cifar_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.Resize(img_size),                          # 32x32 -> 64x64, same size as Q8
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),    #  ImageNet statistics
])
cifar_eval_transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

cifar_train_full = datasets.CIFAR10(root='./data', train=True,  download=True, transform=cifar_train_transform)
cifar_eval_full  = datasets.CIFAR10(root='./data', train=True,  download=True, transform=cifar_eval_transform)
cifar_test_data  = datasets.CIFAR10(root='./data', train=False, download=True, transform=cifar_eval_transform)

# 80/20 train/validation split, same method and seed as Task 1
cifar_rng = np.random.default_rng(2026)
cifar_indices = np.arange(len(cifar_train_full))
cifar_rng.shuffle(cifar_indices)
cifar_split = int(0.8 * len(cifar_indices))
cifar_train_indices = cifar_indices[:cifar_split]          
cifar_val_indices   = cifar_indices[cifar_split:]          

cifar_train_set      = Subset(cifar_train_full, cifar_train_indices)          # augmented
cifar_train_eval_set = Subset(cifar_eval_full,  cifar_train_indices[:10000])  # clean 
cifar_val_set        = Subset(cifar_eval_full,  cifar_val_indices)            # clean

print(f"Train partition: {cifar_train_full.data.shape} {len(cifar_train_full)} images")
print(f"Test partition: {cifar_test_data.data.shape} {len(cifar_test_data)} images")
print(cifar_train_full.classes)
print(cifar_train_set[0][0].shape)     

In [ ]:
# Backbone is frozen and these sets are not augmented -> extract features once
start = time.time()
cifar_val_feats        = extract_features(cifar_val_set)         
cifar_test_feats       = extract_features(cifar_test_data)       
cifar_train_eval_feats = extract_features(cifar_train_eval_set) 
cifar_extract_time = time.time() - start
print(f'feature extraction (one-time): {cifar_extract_time:.1f} s')

In [ ]:
criterion = nn.CrossEntropyLoss()

def train_pretrained(n_fc, train_set, train_eval_feats, val_feats, test_feats,
                     lr=0.05, num_epochs=10, batch_size=64, hidden_units=256):
    torch.manual_seed(2026)
    model = PretrainedResNet(n_fc, hidden_units).to(device)
    optimizer = optim.SGD(model.head.parameters(), lr=lr, momentum=0)   # ONLY the FC layers
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)

    history = {'train': [], 'val': [], 'test': [], 'epoch time': []}
    for epoch in range(num_epochs):
        # ===== Training phase (timed) =====
        start = time.time()
        model.train()
        model.backbone.eval()          # keep BatchNorm running stats frozen too
        for images, labels in loader:
            # 1. to device
            images = images.to(device)
            labels = labels.to(device)
            # 2. zero gradients
            optimizer.zero_grad()
            # 3. forward
            outputs = model(images)
            # 4. loss
            loss = criterion(outputs, labels)
            # 5. backward (gradients only reach the FC layers)
            loss.backward()
            # 6. update the FC layers
            optimizer.step()
        history['epoch time'].append(time.time() - start)

        # ===== Evaluation (head only, on cached features) =====
        history['train'].append(head_accuracy(model.head, train_eval_feats))
        history['val'].append(head_accuracy(model.head, val_feats))
        history['test'].append(head_accuracy(model.head, test_feats))
        print(f"  Epoch [{epoch+1:2d}/{num_epochs}] - Train Acc: {history['train'][-1]:.4f}, "
              f"Val Acc: {history['val'][-1]:.4f}, Test Acc: {history['test'][-1]:.4f}, "
              f"Time: {history['epoch time'][-1]:.1f} s")
    return model, history, n_trainable

In [ ]:
print(f'=== CIFAR-10: ResNet-18 frozen + {best_n_fc} FC layer(s) ===')
cifar_model, cifar_history, _ = train_pretrained(best_n_fc, cifar_train_set,
                                                 cifar_train_eval_feats, cifar_val_feats, cifar_test_feats)

fmnist_history = fc_history[best_n_fc]     # FashionMNIST result from Q8 (same settings)

In [ ]:
class ConvNet(nn.Module):

    def __init__(self, c1=16, c2=32, k=3, in_channels=1, img_size=28):
        super(ConvNet, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, c1, kernel_size=k, stride=1, padding=k//2)  # was Conv2d(1, ...)
        self.conv2 = nn.Conv2d(c1, c2, kernel_size=k, stride=1, padding=k//2)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(c2 * (img_size // 4) ** 2, 256)   # 2 poolings: 28->7 (FashionMNIST), 32->8 (CIFAR)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))    # (N, c1, img/2, img/2)
        x = self.pool(F.relu(self.conv2(x)))    # (N, c2, img/4, img/4)
        x = x.view(x.size(0), -1)               # (N, c2 * (img/4)^2)
        x = F.relu(self.fc1(x))                 # (N, 256)
        return self.fc2(x)                      # (N, 10)

In [ ]:
# CIFAR-10 per-channel mean / std from the training split (same method as Q6, but 3 channels)
pixel_sum = 0.0
pixel_sq_sum = 0.0
N = 0
stat_loader = DataLoader(Subset(datasets.CIFAR10(root='./data', train=True, download=True,
                                                 transform=transforms.ToTensor()), cifar_train_indices),
                         batch_size=1000, shuffle=False)
for data, _ in stat_loader:
    pixel_sum += data.sum(dim=[0, 2, 3])            # one sum per channel -> shape (3,)
    pixel_sq_sum += (data ** 2).sum(dim=[0, 2, 3])
    N += data.numel() / 3                           # pixels PER CHANNEL
cifar_mean = (pixel_sum / N).tolist()
cifar_std  = torch.sqrt(pixel_sq_sum / N - (pixel_sum / N) ** 2).tolist()
print(cifar_mean, cifar_std)

# Trained from scratch -> use CIFAR's own statistics, 32x32, same Q5 augmentation
cnn_cifar_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])
cnn_cifar_eval_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(cifar_mean, cifar_std)])

cnn_cifar_train_set = Subset(datasets.CIFAR10(root='./data', train=True, download=True,
                                              transform=cnn_cifar_train_transform), cifar_train_indices)
cnn_cifar_eval_full = datasets.CIFAR10(root='./data', train=True, download=True, transform=cnn_cifar_eval_transform)
cnn_cifar_train_eval_loader = DataLoader(Subset(cnn_cifar_eval_full, cifar_train_indices[:10000]), batch_size=64, shuffle=False)
cnn_cifar_val_loader  = DataLoader(Subset(cnn_cifar_eval_full, cifar_val_indices), batch_size=64, shuffle=False)
cnn_cifar_test_loader = DataLoader(datasets.CIFAR10(root='./data', train=False, download=True,
                                                    transform=cnn_cifar_eval_transform), batch_size=64, shuffle=False)

best_lr, best_c1, best_c2, best_k, best_bs = 0.1, 32, 64, 3, 32     # best CNN settings from Q6/Q7
torch.manual_seed(2026)
loader = DataLoader(cnn_cifar_train_set, batch_size=best_bs, shuffle=True)
cnn_cifar = ConvNet(c1=best_c1, c2=best_c2, k=best_k, in_channels=3, img_size=32).to(device)
optimizer = optim.SGD(cnn_cifar.parameters(), lr=best_lr, momentum=0)

cnn_cifar_history = {'train': [], 'val': [], 'test': [], 'epoch time': []}
for epoch in range(10):
    # ===== Training phase (timed) =====
    start = time.time()
    cnn_cifar.train()
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = cnn_cifar(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    cnn_cifar_history['epoch time'].append(time.time() - start)

    # ===== Evaluation: clean train subset, validation, test =====
    cnn_cifar.eval()
    for split, eval_loader in [('train', cnn_cifar_train_eval_loader),
                               ('val', cnn_cifar_val_loader), ('test', cnn_cifar_test_loader)]:
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in eval_loader:
                images = images.to(device)
                labels = labels.to(device)
                _, predicted = torch.max(cnn_cifar(images), dim=1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        cnn_cifar_history[split].append(correct / total)
    print(f"  Epoch [{epoch+1:2d}/10] - Train Acc: {cnn_cifar_history['train'][-1]:.4f}, "
          f"Val Acc: {cnn_cifar_history['val'][-1]:.4f}, Test Acc: {cnn_cifar_history['test'][-1]:.4f}")

In [ ]:
q9_df = pd.DataFrame([
    {'dataset': 'FashionMNIST',
     'CNN from scratch': 0.8966,                              # Q7 (with Q5 augmentation)
     'ResNet-18 frozen': fmnist_history['test'][-1]},         # Q8
    {'dataset': 'CIFAR-10',
     'CNN from scratch': cnn_cifar_history['test'][-1],
     'ResNet-18 frozen': cifar_history['test'][-1]},
])
q9_df['transfer gain'] = q9_df['ResNet-18 frozen'] - q9_df['CNN from scratch']
q9_df.round(4)